# LLM Economic Search Agent Verification

This notebook audits whether the economic-search agents receive the right context and whether deterministic gates prevent duplicate simulations.

Default cells are safe: no Brain submission and no LLM calls. Optional live-LLM cells are gated by `RUN_LIVE_LLM = False`.

Important interpretation: the Novelty Critic LLM receives a compact history summary and deterministic risk reports. It does not receive every raw historical expression. Exact duplicate prevention is enforced by `HistoryIndex` plus `ExpressionValidator` before candidates are written or submitted.


In [1]:
from __future__ import annotations

import csv
import json
import os
import sqlite3
import sys
import tempfile
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if not (ROOT / 'alpha_discovery.legacy.economic_search_method').exists():
    for parent in ROOT.parents:
        if (parent / 'alpha_discovery.legacy.economic_search_method').exists():
            ROOT = parent
            break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TARGET_DELAY = 1
RUN_REBUILD_INDEX = False
RUN_LIVE_LLM = False
RUN_CLI_DRY_RUN = True

print('repo root:', ROOT)
print('target delay:', TARGET_DELAY)


repo root: /home/justinjj_/world_quant
target delay: 1


In [3]:
from alpha_discovery.legacy.economic_search_method.cli import main as cli_main
from alpha_discovery.legacy.economic_search_method.critique import DeterministicNoveltyCritic
from alpha_discovery.legacy.economic_search_method.expression_generation import LLMExpressionGenerator
from alpha_discovery.legacy.economic_search_method.expression_validation import AcceptedExpression, ExpressionValidator, windowless_fingerprint
from alpha_discovery.legacy.economic_search_method.field_catalogs import field_catalog_summary, resolve_delay_field_csv
from alpha_discovery.legacy.economic_search_method.history_index import DEFAULT_FIELD_CSV, DEFAULT_INDEX_PATH, DEFAULT_SUMMARY_PATH, HistoryIndex, HistoryIndexer
from alpha_discovery.legacy.economic_search_method.llm_agents import ExpansionAgent, IterationAgent, NoveltyCriticAgent, TheoryProposerAgent, mock_theories
from alpha_discovery.legacy.economic_search_method.portfolio_context import DEFAULT_PORTFOLIO_CONTEXT_PATH, load_portfolio_context
from alpha_discovery.legacy.economic_search_method.portfolio_scoring import score_records_for_admission
from alpha_discovery.legacy.economic_search_method.schemas import TheorySpec
from alpha_discovery.legacy.economic_search_method.semantic_gate import SemanticGateResult, SemanticNoveltyGate, retrieve_semantic_neighbors
from alpha_discovery.legacy.economic_search_method.theory_store import DEFAULT_THEORY_DIR, TheoryStore


## 1. Verify History Index Coverage

This checks that the history index was built from the expected simulation/admission logs. Rebuild only if explicitly requested.


In [4]:
if RUN_REBUILD_INDEX or not DEFAULT_INDEX_PATH.exists() or not DEFAULT_SUMMARY_PATH.exists():
    print('Building history index...')
    summary = HistoryIndexer(index_path=DEFAULT_INDEX_PATH, summary_path=DEFAULT_SUMMARY_PATH, field_csv=DEFAULT_FIELD_CSV).build()
else:
    index = HistoryIndex(index_path=DEFAULT_INDEX_PATH, summary_path=DEFAULT_SUMMARY_PATH)
    if not index.is_compatible():
        raise RuntimeError('History index exists but is not compatible; set RUN_REBUILD_INDEX=True')
    summary = index.summary()

index = HistoryIndex(index_path=DEFAULT_INDEX_PATH, summary_path=DEFAULT_SUMMARY_PATH)
assert index.is_compatible(), 'history index must include delay column'
summary = index.summary()

print('total_records:', summary.get('total_records'))
print('standalone_pass_records:', summary.get('standalone_pass_records'))
print('source paths:')
for path in summary.get('source_paths', []):
    print(' -', path)

required_needles = [
    'artifacts/sim/master.jsonl',
    'artifacts/workshop/workshop_sim.jsonl',
    'alpha_discovery.legacy.llm_grammar_control_method/artifacts/llm_sim.jsonl',
    'artifacts/portfolio_admission/ranked.jsonl',
]
source_blob = '\n'.join(summary.get('source_paths', []))
for needle in required_needles:
    print(f'{needle}:', needle in source_blob)


total_records: 23457
standalone_pass_records: 1535
source paths:
 - /home/justinjj_/world_quant/artifacts/sim/master.jsonl
 - /home/justinjj_/world_quant/artifacts/workshop/workshop_sim.jsonl
 - /home/justinjj_/world_quant/alpha_discovery.legacy.llm_grammar_control_method/artifacts/llm_sim.jsonl
 - /home/justinjj_/world_quant/artifacts/portfolio_admission/ranked.jsonl
artifacts/sim/master.jsonl: True
artifacts/workshop/workshop_sim.jsonl: True
alpha_discovery.legacy.llm_grammar_control_method/artifacts/llm_sim.jsonl: True
artifacts/portfolio_admission/ranked.jsonl: True


In [5]:
with sqlite3.connect(DEFAULT_INDEX_PATH) as conn:
    by_source = conn.execute(
        'SELECT source_path, COUNT(*) FROM records GROUP BY source_path ORDER BY COUNT(*) DESC'
    ).fetchall()
    by_delay = conn.execute(
        'SELECT delay, COUNT(*) FROM records GROUP BY delay ORDER BY delay'
    ).fetchall()

print('records by source:')
for source, count in by_source:
    print(f'{count:>8}  {source}')
print('\nrecords by delay:', by_delay)


records by source:
   20250  /home/justinjj_/world_quant/artifacts/sim/master.jsonl
    1955  /home/justinjj_/world_quant/alpha_discovery.legacy.llm_grammar_control_method/artifacts/llm_sim.jsonl
    1252  /home/justinjj_/world_quant/artifacts/workshop/workshop_sim.jsonl

records by delay: [(1, 23457)]


## 2. Verify Exact Duplicate Blocking Against History

This picks an expression already present in the SQLite index and confirms same-delay validation rejects it before it can be submitted again.


In [6]:
def delay_field_ids(delay: int) -> tuple[Path, set[str]]:
    path = resolve_delay_field_csv(target_delay=delay, fields_csv=DEFAULT_FIELD_CSV)
    ids: set[str] = set()
    with path.open(newline='') as f:
        for row in csv.DictReader(f):
            fid = str(row.get('field_id') or '').strip()
            if fid:
                ids.add(fid)
    return path, ids

def candidate_rows_from_history(delay: int, limit: int = 5000):
    field_path, ids = delay_field_ids(delay)
    with sqlite3.connect(DEFAULT_INDEX_PATH) as conn:
        rows = conn.execute(
            'SELECT expression, delay, fields_json, skeleton_json, source_path FROM records WHERE delay=? ORDER BY id DESC LIMIT ?',
            (delay, limit),
        ).fetchall()
    for expression, row_delay, fields_json, skeleton_json, source_path in rows:
        fields = json.loads(fields_json or '[]')
        skeleton = json.loads(skeleton_json or '[]')
        if fields and set(fields).issubset(ids) and '=' not in expression:
            yield {
                'expression': expression,
                'delay': row_delay,
                'fields': fields,
                'skeleton': skeleton,
                'source_path': source_path,
                'field_path': field_path,
            }

duplicate_sample = None
duplicate_validation = None
for sample in candidate_rows_from_history(TARGET_DELAY):
    theory = TheorySpec.from_dict({
        'theory_id': 'history_duplicate_probe',
        'title': 'History duplicate probe',
        'mechanism': 'Deliberately test exact duplicate rejection.',
        'data_roles': {'signal': ' '.join(sample['fields'][:2])},
        'expected_sign': 'positive',
        'horizon_days': [20],
        'field_cluster_targets': sample['fields'][:4],
        'novelty_claim': 'This should be rejected as a duplicate.',
        'kill_criteria': ['duplicate'],
        'expand_criteria': ['never'],
    })
    candidate = {
        'expression': sample['expression'],
        'field_roles': {'signal': sample['fields'][0]},
        'mechanism_translation': 'Exact expression copied from history.',
        'expected_sign': 'positive',
        'horizon_days': 20,
        'novelty_argument': 'none',
        'risk_notes': [],
        'generation_source': 'verification_notebook',
    }
    validator = ExpressionValidator(field_csv=sample['field_path'], history_index=index, target_delay=TARGET_DELAY)
    validation = validator.validate_batch(theory=theory, candidates=[candidate], requested_count=1)
    if validation.rejected and validation.rejected[0].get('reason') == 'duplicate_expression_same_delay':
        duplicate_sample = sample
        duplicate_validation = validation
        break

assert duplicate_sample is not None, 'Could not find a same-delay historical expression that validates far enough to hit duplicate gate.'
print('duplicate source:', duplicate_sample['source_path'])
print('duplicate fields:', duplicate_sample['fields'])
print('duplicate skeleton:', duplicate_sample['skeleton'])
print('duplicate expression:', duplicate_sample['expression'][:300])
pprint(duplicate_validation.rejected[0])


duplicate source: /home/justinjj_/world_quant/alpha_discovery.legacy.llm_grammar_control_method/artifacts/llm_sim.jsonl
duplicate fields: ['anl4_ptp_number']
duplicate skeleton: ['group_backfill', 'group_backfill', 'abs']
duplicate expression: group_backfill(group_backfill(abs(anl4_ptp_number), industry, 10), sector, 20)
{'candidate': {'expected_sign': 'positive',
               'expression': 'group_backfill(group_backfill(abs(anl4_ptp_number), '
                             'industry, 10), sector, 20)',
               'field_roles': {'signal': 'anl4_ptp_number'},
               'generation_source': 'verification_notebook',
               'horizon_days': 20,
               'mechanism_translation': 'Exact expression copied from history.',
               'novelty_argument': 'none',
               'risk_notes': []},
 'expression': 'group_backfill(group_backfill(abs(anl4_ptp_number), industry, '
               '10), sector, 20)',
 'reason': 'duplicate_expression_same_delay'}


In [7]:
risk = index.jaccard_risk(
    duplicate_sample['fields'],
    duplicate_sample['skeleton'],
    target_delay=TARGET_DELAY,
)
pprint(risk)
assert risk['high'] or risk['max_jaccard'] >= 0.85, 'Exact field/skeleton should look high-risk in recent history.'


{'high': True, 'max_jaccard': 1.0}


## 3. Verify Novelty Critic Context

This uses a capture client, not a live LLM. It proves what payload the Novelty Critic receives. The critic sees compact `history_summary`, deterministic reports, and current portfolio context.


In [8]:
class CaptureJSONClient:
    def __init__(self, response):
        self.response = response
        self.calls = []
        self.last_raw_text = json.dumps(response)
        self.last_stop_reason = 'capture'

    def json_call(self, *, system: str, user: str):
        payload = json.loads(user)
        self.calls.append({'system': system, 'user': payload})
        return self.response

portfolio_context = load_portfolio_context(DEFAULT_PORTFOLIO_CONTEXT_PATH)
test_theory = TheorySpec.from_dict({
    'theory_id': 'duplicate_fundamental_profitability_probe',
    'title': 'Duplicate fundamental profitability probe',
    'mechanism': 'Operating income, assets, cap, returns and industry grouping repeat submitted fundamental profitability incentives.',
    'data_roles': {'signal': 'fundamental6 operating_income cap assets', 'condition': 'pv1 returns industry'},
    'expected_sign': 'positive profitability is long',
    'horizon_days': [20, 60],
    'field_cluster_targets': ['fundamental6', 'operating_income', 'assets', 'cap', 'returns'],
    'novelty_claim': 'This is intentionally weak and should be criticized.',
    'kill_criteria': ['duplicate'],
    'expand_criteria': ['none'],
})

det_critic = DeterministicNoveltyCritic(index, portfolio_context=portfolio_context, target_delay=TARGET_DELAY)
deterministic_reports = {test_theory.theory_id: det_critic.critique(test_theory)}
capture = CaptureJSONClient({test_theory.theory_id: {'decision': 'REJECT', 'reason': 'capture response'}})
critic = NoveltyCriticAgent(client=capture)
captured_decisions = critic.critique(
    theories=[test_theory],
    deterministic_reports=deterministic_reports,
    history_summary=summary,
    portfolio_context=portfolio_context,
    target_delay=TARGET_DELAY,
)
critic_payload = capture.calls[-1]['user']

print('captured decision:', captured_decisions)
print('payload keys:', sorted(critic_payload))
print('history_summary keys:', sorted(critic_payload['history_summary']))
print('history total_records in LLM payload:', critic_payload['history_summary'].get('total_records'))
print('index total_records:', summary.get('total_records'))
assert critic_payload['history_summary'].get('total_records') == summary.get('total_records')
assert 'deterministic_reports' in critic_payload
assert 'current_submitted_portfolio' in critic_payload
pprint(deterministic_reports[test_theory.theory_id])


captured decision: {'duplicate_fundamental_profitability_probe': {'decision': 'REJECT', 'reason': 'capture response'}}
payload keys: ['current_submitted_portfolio', 'delay_definition', 'deterministic_reports', 'history_summary', 'target_delay', 'theories']
history_summary keys: ['admission', 'category', 'dataset', 'delay', 'failed_checks', 'field', 'high_corr_by_dataset', 'operator', 'pool_impact', 'standalone_pass_records', 'total_records']
history total_records in LLM payload: 23457
index total_records: 23457
{'decision': 'REJECT',
 'portfolio_overlap': ['submitted field exposure: industry',
                       'submitted field exposure: returns',
                       'submitted field exposure: assets',
                       'submitted field exposure: operating_income',
                       'fundamental+price_volume | debt_lt+equity+industry | '
                       'group_scale>rank>days_from_last_change',
                       'fundamental+price_volume | '
              

## 4. Verify Proposer, Generator, Iterate, and Expand Payloads

These checks do not call the LLM. They inspect prompts and constructed payloads.


In [9]:
prompt_paths = {
    'proposer': ROOT / 'alpha_discovery.legacy.economic_search_method/prompts/theory_proposer_system.txt',
    'critic': ROOT / 'alpha_discovery.legacy.economic_search_method/prompts/novelty_critic_system.txt',
    'generator': ROOT / 'alpha_discovery.legacy.economic_search_method/prompts/candidate_generator_system.txt',
    'iteration': ROOT / 'alpha_discovery.legacy.economic_search_method/prompts/iteration_agent_system.txt',
    'expansion': ROOT / 'alpha_discovery.legacy.economic_search_method/prompts/expansion_agent_system.txt',
}
for name, path in prompt_paths.items():
    text = path.read_text()
    print(f'[{name}]', text.splitlines()[0])
    assert 'JSON' in text or 'json' in text

assert 'market dynamic' in prompt_paths['proposer'].read_text()
assert 'novel alpha' in prompt_paths['critic'].read_text()
assert 'alpha digging expert' in prompt_paths['generator'].read_text()
assert 'portfolio-contribution expert' in prompt_paths['iteration'].read_text()
assert 'portfolio-useful diversification' in prompt_paths['expansion'].read_text()


[proposer] you are an expert in understanding market dynamic and coming up with possible theory explaining market dynamics.
[critic] you are an expert in identifying novel alpha.
[generator] you are an alpha digging expert.
[iteration] You are an alpha repair and portfolio-contribution expert.
[expansion] You are an alpha expansion expert focused on portfolio-useful diversification.


In [11]:
store = TheoryStore(DEFAULT_THEORY_DIR)
field_path = resolve_delay_field_csv(target_delay=TARGET_DELAY, fields_csv=DEFAULT_FIELD_CSV)
field_summary = field_catalog_summary(field_path, target_delay=TARGET_DELAY)
theory = None
for candidate_id in ['option9_oi_buildup_pv13_peer_lag', 'option8_term_structure_analyst_horizon']:
    try:
        theory = store.load_theory(candidate_id)
        break
    except FileNotFoundError:
        pass
if theory is None:
    theory = mock_theories(n=1)[0]

generator = LLMExpressionGenerator(field_csv=field_path, history_index=index, target_delay=TARGET_DELAY)
generator_payloads = {}
for mode in ['probe', 'iterate', 'expand']:
    gen_payload = generator._request_payload(
        theory=theory,
        stage=mode,
        generation_mode=mode,
        requested=6,
        portfolio_context=portfolio_context,
        history_summary=summary,
        field_catalog_summary=field_summary,
        prior_results=store.all_results(theory.theory_id, target_delay=TARGET_DELAY),
        existing_expressions=sorted(store.theory_candidate_expressions(theory.theory_id, target_delay=TARGET_DELAY))[-10:],
        branch_decision=store.read_decision(theory.theory_id, target_delay=TARGET_DELAY),
    )
    generator_payloads[mode] = gen_payload
    print('\nmode:', mode)
    print('payload keys:', sorted(gen_payload.keys()))
    print('portfolio objective:', gen_payload['portfolio_objective']['objective'])
    print('prior_result_summary keys:', sorted(gen_payload['prior_result_summary'].keys()))
    assert gen_payload['generation_mode'] == mode
    assert 'portfolio_objective' in gen_payload
    assert 'prior_result_summary' in gen_payload
    assert 'current_submitted_portfolio' in gen_payload



mode: probe
payload keys: ['brain_grammar_context', 'branch_decision', 'current_submitted_portfolio', 'delay_definition', 'field_catalog_sample', 'field_catalog_summary', 'generation_mode', 'history_summary', 'novelty_rules', 'output_contract', 'portfolio_objective', 'prior_result_summary', 'prior_theory_expressions', 'prior_theory_results', 'requested_count', 'role_candidate_fields', 'stage', 'target_delay', 'task', 'theory']
portfolio objective: Generate candidates likely to pass standalone checks and improve same-delay equal-weight pool contribution.
prior_result_summary keys: ['admission_decisions', 'best', 'failed_checks', 'n_done', 'pool_hurt_examples', 'positive_pool_examples', 'worst']

mode: iterate
payload keys: ['brain_grammar_context', 'branch_decision', 'current_submitted_portfolio', 'delay_definition', 'field_catalog_sample', 'field_catalog_summary', 'generation_mode', 'history_summary', 'novelty_rules', 'output_contract', 'portfolio_objective', 'prior_result_summary', '

## 5. Verify Portfolio Scoring Semantics With Synthetic PnLs

This proves same-delay pool size and global-correlation pool size are different things.


In [12]:
dates = [f'2023-01-{day:02d}' for day in range(1, 31)]
portfolio = {
    'd0_alpha': {
        'delay': 0,
        'settings': {'delay': 0},
        'pnl': {date: float(i) for i, date in enumerate(dates)},
        'is_sharpe': 1.5,
        'is_fitness': 1.2,
    },
    'd1_alpha': {
        'delay': 1,
        'settings': {'delay': 1},
        'pnl': {date: float(30 - i) for i, date in enumerate(dates)},
        'is_sharpe': 1.4,
        'is_fitness': 1.1,
    },
}
row = {
    'alpha_id': 'candidate_d0',
    'candidate_id': 'c1',
    'theory_id': 'synthetic_pool_test',
    'expression': 'rank(x)',
    'status': 'done',
    'settings': {'delay': 0},
    'is_sharpe': 1.3,
    'checks': [{'name': 'LOW_SHARPE', 'result': 'PASS'}],
}
scored = score_records_for_admission(
    [row],
    portfolio,
    candidate_pnls={'candidate_d0': {date: float(i * 2) for i, date in enumerate(dates)}},
)
pprint(scored[0])
assert scored[0]['admission_same_delay_pool_size'] == 1
assert scored[0]['admission_global_pool_size'] == 2
assert 'admission_delta_pool_sharpe' in scored[0]
assert 'admission_residual_sharpe' in scored[0]
assert 'admission_decision' in scored[0]


{'admission': {'candidate_pnl_available': True,
               'corr_gate_evaluated': True,
               'corr_gate_pass': True,
               'decision': 'WATCH',
               'delay': 0,
               'delta_pool_sharpe': None,
               'global_max_corr': None,
               'global_pool_size': 2,
               'max_corr_alpha_fitness': None,
               'max_corr_alpha_id': None,
               'max_corr_alpha_sharpe': None,
               'pool_corr': None,
               'pool_sharpe_after': None,
               'pool_sharpe_before': None,
               'portfolio_available': True,
               'reason': 'DELTA_UNAVAILABLE',
               'residual_sharpe': None,
               'same_delay_pool_size': 1,
               'shared_days': 29},
 'admission_candidate_pnl_available': True,
 'admission_corr_gate_evaluated': True,
 'admission_corr_gate_pass': True,
 'admission_decision': 'WATCH',
 'admission_delay': 0,
 'admission_delta_pool_sharpe': None,
 'admission_g

## 6. Optional CLI Dry Run For `advance`

This runs in a temporary directory, uses mock generation, does not call Brain, and does not call the LLM.


In [13]:
if RUN_CLI_DRY_RUN:
    with tempfile.TemporaryDirectory() as td:
        root = Path(td)
        artifacts = root / 'artifacts'
        delay_dir = root / 'field_catalogs'
        delay_dir.mkdir()
        fields = delay_dir / 'fields_delay1_TOP3000.csv'
        fields.write_text(
            'dataset_id,field_id,field_name,type,coverage,category,subcategory,alpha_count,user_count,delay,universe\n'
            'analyst4,eps_rev,EPS Revision,MATRIX,0.9,Analyst,Est,10,1,1,TOP3000\n'
            'news12,news_sent,News Sentiment,MATRIX,0.8,News,Sent,3,1,1,TOP3000\n'
            'pv1,returns,Returns,MATRIX,1.0,Price Volume,PV,100,1,1,TOP3000\n'
        )
        idx = root / 'idx.sqlite'
        summ = root / 'summary.json'
        HistoryIndexer(index_path=idx, summary_path=summ, field_csv=fields, log_paths=[]).build()
        temp_store = TheoryStore(artifacts / 'theories')
        for theory_id, decision in [('dry_iter', 'ITERATE'), ('dry_expand', 'EXPAND'), ('dry_kill', 'KILL')]:
            t = TheorySpec.from_dict({
                'theory_id': theory_id,
                'title': theory_id,
                'mechanism': 'revision confirmed by news',
                'data_roles': {'signal': 'analyst4 eps revision', 'condition': 'news12 sentiment'},
                'expected_sign': 'positive',
                'horizon_days': [20],
                'field_cluster_targets': ['analyst4', 'news12'],
                'novelty_claim': 'test',
                'kill_criteria': ['kill'],
                'expand_criteria': ['expand'],
                'status': 'ACCEPTED',
            })
            temp_store.save_theory(t)
            temp_store.write_decision(theory_id, {'decision': decision, 'reason': 'dry run'}, target_delay=1)
        (root / 'portfolio.json').write_text('{}')
        rc = cli_main([
            '--artifacts-dir', str(artifacts),
            '--theory-dir', str(artifacts / 'theories'),
            '--index-path', str(idx),
            '--summary-path', str(summ),
            '--fields-csv', str(fields),
            '--delay-fields-dir', str(delay_dir),
            '--portfolio-context-path', str(root / 'portfolio.json'),
            'advance', '--delay', '1', '--mock', '--skip-portfolio', '--no-admission-score',
        ])
        print('advance rc:', rc)
        assert rc == 0
        assert list((artifacts / 'theories' / 'dry_iter').glob('candidates_iterate_*_delay1.jsonl'))
        assert list((artifacts / 'theories' / 'dry_expand').glob('candidates_expand_*_delay1.jsonl'))
        assert not list((artifacts / 'theories' / 'dry_kill').glob('candidates_*_delay1.jsonl'))
else:
    print('CLI dry run disabled')


ADVANCE: using existing portfolio context (0 submitted)
ADVANCE: selected 1 iterate and 1 expand theories for delay=1
dry_iter: wrote 12 LLM candidates to /tmp/tmp33jfy6yl/artifacts/theories/dry_iter/candidates_iterate_001_delay1.jsonl (rejected=0, skeletons=12)
Wrote combined plan: /tmp/tmp33jfy6yl/artifacts/candidate_plans/economic_plan_delay1_iterate_20260416_231012.jsonl
Generated candidates: 12
dry_expand: wrote 12 LLM candidates to /tmp/tmp33jfy6yl/artifacts/theories/dry_expand/candidates_expand_001_delay1.jsonl (rejected=0, skeletons=12)
Wrote combined plan: /tmp/tmp33jfy6yl/artifacts/candidate_plans/economic_plan_delay1_expand_20260416_231012.jsonl
Generated candidates: 12
Simulation skipped. Re-run advance with --simulate to send second-stage plans to Brain.
advance rc: 0


## 7. Scope Audit: What Context Does Each Agent Get?

This summarizes the actual context scope. The anti-duplication guarantee is split: LLM agents receive compact history/portfolio context, while deterministic validators query the SQLite history index for exact expression and field/skeleton duplicates.


In [15]:
proposer_capture = CaptureJSONClient([mock_theories(n=1)[0].to_dict()])
captured_proposals = TheoryProposerAgent(client=proposer_capture).propose(
    history_summary=summary,
    existing_theories=store.list_theories(),
    portfolio_context=portfolio_context,
    target_delay=TARGET_DELAY,
    field_catalog_summary=field_summary,
    sprint_focus='scope audit',
    n=1,
)
proposer_payload = proposer_capture.calls[-1]['user']

gen_payload = generator._request_payload(
    theory=theory,
    stage='probe',
    generation_mode='probe',
    requested=4,
    portfolio_context=portfolio_context,
    history_summary=summary,
    field_catalog_summary=field_summary,
    prior_results=store.all_results(theory.theory_id, target_delay=TARGET_DELAY),
    existing_expressions=sorted(store.theory_candidate_expressions(theory.theory_id, target_delay=TARGET_DELAY))[-10:],
    branch_decision=store.read_decision(theory.theory_id, target_delay=TARGET_DELAY),
)

scope = {
    'proposer': {
        'payload_keys': sorted(proposer_payload.keys()),
        'history_total_records': proposer_payload['history_summary'].get('total_records'),
        'portfolio_n_submitted': proposer_payload['current_submitted_portfolio'].get('n_submitted'),
        'same_delay_submitted': len(proposer_payload['current_submitted_portfolio'].get('same_delay_submitted_alphas', [])),
    },
    'novelty_critic': {
        'payload_keys': sorted(critic_payload.keys()),
        'history_total_records': critic_payload['history_summary'].get('total_records'),
        'has_deterministic_reports': 'deterministic_reports' in critic_payload,
        'has_raw_expression_history': False,
    },
    'expression_generator': {
        'payload_keys': sorted(gen_payload.keys()),
        'prior_theory_expression_count': len(gen_payload.get('prior_theory_expressions', [])),
        'has_prior_result_summary': 'prior_result_summary' in gen_payload,
        'has_portfolio_objective': 'portfolio_objective' in gen_payload,
    },
    'deterministic_duplicate_gate': {
        'source': str(DEFAULT_INDEX_PATH),
        'exact_expression_lookup': True,
        'field_skeleton_lookup': True,
        'same_delay_aware': True,
    },
}
pprint(scope)

assert scope['proposer']['history_total_records'] == summary.get('total_records')
assert scope['novelty_critic']['history_total_records'] == summary.get('total_records')
assert scope['novelty_critic']['has_deterministic_reports']
assert scope['expression_generator']['has_portfolio_objective']


{'deterministic_duplicate_gate': {'exact_expression_lookup': True,
                                  'field_skeleton_lookup': True,
                                  'same_delay_aware': True,
                                  'source': '/home/justinjj_/world_quant/alpha_discovery.legacy.economic_search_method/artifacts/history_index.sqlite'},
 'expression_generator': {'has_portfolio_objective': True,
                          'has_prior_result_summary': True,
                          'payload_keys': ['brain_grammar_context',
                                           'branch_decision',
                                           'current_submitted_portfolio',
                                           'delay_definition',
                                           'field_catalog_sample',
                                           'field_catalog_summary',
                                           'generation_mode',
                                           'history_summary',
          

## 8. Grammar-Lane Duplicate Veto

This specifically pulls an already simulated grammar-lane expression from `artifacts/sim/master.jsonl` and confirms the same-delay validator vetoes it. This is the check that prevents retesting old grammar expressions even if an LLM proposes them later.


In [16]:
def find_grammar_lane_veto(delay: int, limit: int = 10000):
    field_path, ids = delay_field_ids(delay)
    with sqlite3.connect(DEFAULT_INDEX_PATH) as conn:
        rows = conn.execute(
            '''
            SELECT expression, delay, fields_json, skeleton_json, source_path, search_lane
            FROM records
            WHERE delay=? AND search_lane='grammar'
            ORDER BY id DESC
            LIMIT ?
            ''',
            (delay, limit),
        ).fetchall()
    for expression, row_delay, fields_json, skeleton_json, source_path, lane in rows:
        fields = json.loads(fields_json or '[]')
        skeleton = json.loads(skeleton_json or '[]')
        if not fields or not set(fields).issubset(ids) or '=' in expression:
            continue
        t = TheorySpec.from_dict({
            'theory_id': 'grammar_lane_duplicate_probe',
            'title': 'Grammar lane duplicate probe',
            'mechanism': 'Deliberately copy one grammar-lane expression to verify veto.',
            'data_roles': {'signal': ' '.join(fields[:2])},
            'expected_sign': 'positive',
            'horizon_days': [20],
            'field_cluster_targets': fields[:4],
            'novelty_claim': 'This should be rejected as already simulated.',
            'kill_criteria': ['duplicate'],
            'expand_criteria': ['never'],
        })
        c = {
            'expression': expression,
            'field_roles': {'signal': fields[0]},
            'mechanism_translation': 'Exact expression copied from grammar lane history.',
            'expected_sign': 'positive',
            'horizon_days': 20,
            'novelty_argument': 'none',
            'risk_notes': [],
            'generation_source': 'verification_notebook',
        }
        v = ExpressionValidator(field_csv=field_path, history_index=index, target_delay=delay).validate_batch(
            theory=t,
            candidates=[c],
            requested_count=1,
        )
        if v.rejected and v.rejected[0].get('reason') == 'duplicate_expression_same_delay':
            return {
                'expression': expression,
                'fields': fields,
                'skeleton': skeleton,
                'source_path': source_path,
                'search_lane': lane,
                'validation': v,
            }
    raise AssertionError('No grammar-lane sample reached duplicate_expression_same_delay veto.')

grammar_veto = find_grammar_lane_veto(TARGET_DELAY)
print('source:', grammar_veto['source_path'])
print('lane:', grammar_veto['search_lane'])
print('fields:', grammar_veto['fields'])
print('skeleton:', grammar_veto['skeleton'])
print('expression:', grammar_veto['expression'][:300])
pprint(grammar_veto['validation'].rejected[0])
assert grammar_veto['validation'].rejected[0]['reason'] == 'duplicate_expression_same_delay'


source: /home/justinjj_/world_quant/artifacts/sim/master.jsonl
lane: grammar
fields: ['pv13_hierarchy_min2_focused_pureplay_513_sector']
skeleton: ['group_rank', 'ts_product', 'group_neutralize']
expression: group_rank(ts_product(group_neutralize(pv13_hierarchy_min2_focused_pureplay_513_sector, subindustry), 200), industry)
{'candidate': {'expected_sign': 'positive',
               'expression': 'group_rank(ts_product(group_neutralize(pv13_hierarchy_min2_focused_pureplay_513_sector, '
                             'subindustry), 200), industry)',
               'field_roles': {'signal': 'pv13_hierarchy_min2_focused_pureplay_513_sector'},
               'generation_source': 'verification_notebook',
               'horizon_days': 20,
               'mechanism_translation': 'Exact expression copied from grammar '
                                        'lane history.',
               'novelty_argument': 'none',
               'risk_notes': []},
 'expression': 'group_rank(ts_product(group_n

## 9. Pipeline Torture Harness: Deterministic Hard Gates

This pounds the deterministic layer from multiple angles. These checks should never need an LLM: exact history duplicates, whitespace mutations, neutral wrappers, window-only mutations, malformed expressions, unknown operators/fields, missing required datasets, duplicate-in-batch, sign repair, and cross-delay retest semantics.


In [18]:
import copy
import re
from types import SimpleNamespace

STRESS_REPORT = {}


def stress_theory(theory_id='pipeline_torture', fields=None, mechanism='Pipeline torture probe'):
    fields = fields or []
    return TheorySpec.from_dict({
        'theory_id': theory_id,
        'title': theory_id.replace('_', ' ').title(),
        'mechanism': mechanism,
        'data_roles': {'signal': ' '.join(fields) or 'generic signal'},
        'expected_sign': 'positive',
        'horizon_days': [20],
        'field_cluster_targets': fields[:4],
        'novelty_claim': 'Stress test candidate.',
        'kill_criteria': ['duplicate or invalid'],
        'expand_criteria': ['novel and useful'],
    })


def validator_for(delay=TARGET_DELAY):
    fp, _ = delay_field_ids(delay)
    return ExpressionValidator(field_csv=fp, history_index=index, target_delay=delay)


def validate_expr(expr: str, *, theory=None, delay: int = TARGET_DELAY):
    theory = theory or stress_theory(fields=grammar_veto.get('fields', [])[:4])
    v = validator_for(delay).validate_batch(
        theory=theory,
        candidates=[{'expression': expr, 'field_roles': {}}],
        requested_count=1,
    )
    if v.rejected:
        row = v.rejected[0]
        return {'status': 'REJECT', 'reason': row.get('reason'), 'details': row.get('details')}
    row = v.accepted[0]
    return {
        'status': 'ACCEPT',
        'reason': None,
        'details': {
            'novelty_score': row.novelty_score,
            'fields': row.fields,
            'skeleton': row.operator_skeleton,
            'validation_notes': row.validation_notes,
            'cross_delay_retest': row.cross_delay_retest,
        },
    }


def whitespace_mutation(expr: str) -> str:
    return re.sub(r'\s+', ' ', expr.replace('(', ' ( ').replace(')', ' ) ').replace(',', ' , ')).strip()


def tweak_first_ts_window(expr: str) -> str | None:
    def repl(m):
        return f'{m.group(1)}{int(m.group(2)) + 1}{m.group(3)}'
    out, n = re.subn(r'(ts_[A-Za-z_]+\([^()]*?,\s*)(\d+)(\s*\))', repl, expr, count=1)
    return out if n else None


def field_id_by_dataset(include_dataset=None, exclude_dataset=None):
    with open(field_path, newline='') as f:
        for row in csv.DictReader(f):
            dataset = (row.get('dataset_id') or '').lower()
            fid = row.get('field_id')
            if not fid:
                continue
            if include_dataset and dataset != include_dataset.lower():
                continue
            if exclude_dataset and dataset == exclude_dataset.lower():
                continue
            return fid
    raise AssertionError(f'No field found include={include_dataset} exclude={exclude_dataset}')


def find_window_mutation_sample(delay: int = TARGET_DELAY):
    with sqlite3.connect(DEFAULT_INDEX_PATH) as conn:
        rows = conn.execute(
            """
            SELECT expression, fields_json, skeleton_json, source_path, search_lane
            FROM records
            WHERE delay=? AND search_lane='grammar' AND expression LIKE '%ts_%'
            ORDER BY id DESC
            LIMIT 10000
            """,
            (delay,),
        ).fetchall()
    for expression, fields_json, skeleton_json, source_path, lane in rows:
        mutated = tweak_first_ts_window(expression)
        if not mutated:
            continue
        exact = validate_expr(expression, delay=delay)
        if exact['reason'] == 'duplicate_expression_same_delay':
            return {
                'expression': expression,
                'mutated': mutated,
                'fields': json.loads(fields_json or '[]'),
                'skeleton': json.loads(skeleton_json or '[]'),
                'source_path': source_path,
                'search_lane': lane,
            }
    raise AssertionError('No window-mutable grammar-lane duplicate found.')


def find_deterministic_pass_candidate(limit_fields: int = 140):
    field_ids = [row['field_id'] for row in field_summary.get('field_examples', []) if row.get('field_id')]
    if len(field_ids) < limit_fields:
        with open(field_path, newline='') as f:
            for row in csv.DictReader(f):
                fid = row.get('field_id')
                if fid and fid not in field_ids:
                    field_ids.append(fid)
                if len(field_ids) >= limit_fields:
                    break
    t = stress_theory('deterministic_pass_probe')
    v = validator_for(TARGET_DELAY)
    for fid in field_ids:
        for expr in [
            f'rank(ts_delta({fid}, 3))',
            f'ts_rank({fid}, 17)',
            f'group_neutralize(rank({fid}), subindustry)',
            f'group_rank(ts_zscore({fid}, 13), industry)',
        ]:
            batch = v.validate_batch(
                theory=t,
                candidates=[{'expression': expr, 'field_roles': {'signal': fid}}],
                requested_count=1,
            )
            if batch.accepted:
                return t, batch.accepted[0]
    raise AssertionError('Could not find a deterministic-pass candidate for downstream stress tests.')


base_expr = grammar_veto['expression']
window_sample = find_window_mutation_sample(TARGET_DELAY)
non_pv_field = field_id_by_dataset(exclude_dataset='pv13')
required_pv13_theory = TheorySpec.from_dict({
    'theory_id': 'requires_pv13_missing_dataset_probe',
    'title': 'Requires PV13 Missing Dataset Probe',
    'mechanism': 'Must use pv13 relationship data.',
    'data_roles': {'signal': 'pv13 relationship peer signal'},
    'expected_sign': 'positive',
    'horizon_days': [20],
    'field_cluster_targets': ['pv13 relationship'],
    'novelty_claim': 'Should reject candidates without pv13.',
    'kill_criteria': ['missing pv13'],
    'expand_criteria': ['uses pv13'],
})

deterministic_cases = {
    'exact_history_copy': (base_expr, {'duplicate_expression_same_delay'}, None),
    'whitespace_only_history_copy': (whitespace_mutation(base_expr), {'duplicate_expression_same_delay'}, None),
    'neutral_add_zero_wrapper': (f'add({base_expr}, 0)', {'trivial_neutral_wrapper_of_prior_expression'}, None),
    'neutral_multiply_one_wrapper': (f'multiply({base_expr}, 1)', {'trivial_neutral_wrapper_of_prior_expression'}, None),
    'neutral_divide_one_wrapper': (f'divide({base_expr}, 1)', {'trivial_neutral_wrapper_of_prior_expression'}, None),
    'nested_neutral_wrapper': (f'add(multiply({base_expr}, 1), 0)', {'trivial_neutral_wrapper_of_prior_expression'}, None),
    'window_exact_copy': (window_sample['expression'], {'duplicate_expression_same_delay'}, None),
    'window_plus_one': (window_sample['mutated'], {'field_skeleton_seen_in_history', 'high_field_operator_jaccard_history'}, None),
    'window_plus_one_whitespace': (whitespace_mutation(window_sample['mutated']), {'field_skeleton_seen_in_history', 'high_field_operator_jaccard_history'}, None),
    'unknown_operator': ('totally_fake_operator(rank(close), 5)', {'unknown_operator', 'unknown_identifier_or_field'}, None),
    'unknown_field': ('rank(ts_delta(not_a_real_field_12345, 5))', {'unknown_identifier_or_field'}, None),
    'unbalanced_parentheses': ('rank(ts_delta(not_a_real_field_12345, 5)', {'unbalanced_parentheses'}, None),
    'missing_required_dataset': (f'rank(ts_delta({non_pv_field}, 3))', {'missing_required_theory_dataset'}, required_pv13_theory),
}

stress_results = {}
for name, (expr, expected_reasons, theory_override) in deterministic_cases.items():
    result = validate_expr(expr, theory=theory_override, delay=TARGET_DELAY)
    stress_results[name] = result
    print(name, '=>', result)
    print('  expr:', expr[:260])
    assert result['status'] == 'REJECT', f'{name} unexpectedly survived: {result}'
    assert result['reason'] in expected_reasons, f'{name} rejected for unexpected reason: {result}'

sign_flip_cases = {
    'sign_flip_reverse_wrapper': f'reverse({base_expr})',
    'sign_flip_multiply_minus_one': f'multiply({base_expr}, -1)',
}
for name, expr in sign_flip_cases.items():
    result = validate_expr(expr, delay=TARGET_DELAY)
    stress_results[name] = result
    print(name, '=>', result)
    assert result['status'] == 'ACCEPT', f'{name} should survive as a low-novelty sign repair: {result}'
    assert 'sign_flip_of_prior_expression' in result['details'].get('validation_notes', [])
    assert result['details'].get('novelty_score', 1.0) <= 0.25

semantic_theory, semantic_candidate = find_deterministic_pass_candidate()
validator = validator_for(TARGET_DELAY)
duplicate_batch = validator.validate_batch(
    theory=semantic_theory,
    candidates=[
        {'expression': semantic_candidate.expression, 'field_roles': semantic_candidate.field_roles},
        {'expression': semantic_candidate.expression, 'field_roles': semantic_candidate.field_roles},
    ],
    requested_count=2,
)
print('duplicate_in_llm_batch accepted/rejected:', len(duplicate_batch.accepted), duplicate_batch.rejected)
assert len(duplicate_batch.accepted) == 1
assert any(row['reason'] == 'duplicate_in_llm_batch' for row in duplicate_batch.rejected)

with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    fields_tmp = root / 'fields.csv'
    fields_tmp.write_text(
        'dataset_id,field_id,field_name,type,coverage,category,subcategory,alpha_count,user_count,delay,universe\n'
        'news12,news_sent,News Sentiment,MATRIX,0.8,News,Sent,3,1,0,TOP3000\n'
        'news12,news_sent,News Sentiment,MATRIX,0.8,News,Sent,3,1,1,TOP3000\n'
    )
    expr = 'rank(ts_delta(news_sent, 5))'
    log = root / 'sim.jsonl'
    log.write_text(json.dumps({'expression': expr, 'fields': ['news_sent'], 'settings': {'delay': 1}, 'status': 'done'}) + '\n')
    HistoryIndexer(index_path=root / 'idx.sqlite', summary_path=root / 'summary.json', field_csv=fields_tmp, log_paths=[log]).build()
    temp_index = HistoryIndex(index_path=root / 'idx.sqlite', summary_path=root / 'summary.json')
    cross_theory = stress_theory('cross_delay_retest_probe', fields=['news_sent'])
    cross = ExpressionValidator(field_csv=fields_tmp, history_index=temp_index, target_delay=0).validate_batch(
        theory=cross_theory,
        candidates=[{'expression': expr, 'field_roles': {'signal': 'news_sent'}}],
        requested_count=1,
    )
    print('cross_delay_retest accepted:', len(cross.accepted), cross.accepted[0].cross_delay_retest if cross.accepted else None)
    assert len(cross.accepted) == 1
    assert cross.accepted[0].cross_delay_retest

STRESS_REPORT['deterministic_hard_gates'] = {
    'cases': stress_results,
    'duplicate_batch_rejections': duplicate_batch.rejected,
    'deterministic_pass_expression': semantic_candidate.expression,
}
print('deterministic stress cases passed:', len(stress_results) + 2)


exact_history_copy => {'status': 'REJECT', 'reason': 'duplicate_expression_same_delay', 'details': None}
  expr: group_rank(ts_product(group_neutralize(pv13_hierarchy_min2_focused_pureplay_513_sector, subindustry), 200), industry)
whitespace_only_history_copy => {'status': 'REJECT', 'reason': 'duplicate_expression_same_delay', 'details': None}
  expr: group_rank ( ts_product ( group_neutralize ( pv13_hierarchy_min2_focused_pureplay_513_sector , subindustry ) , 200 ) , industry )
neutral_add_zero_wrapper => {'status': 'REJECT', 'reason': 'trivial_neutral_wrapper_of_prior_expression', 'details': None}
  expr: add(group_rank(ts_product(group_neutralize(pv13_hierarchy_min2_focused_pureplay_513_sector, subindustry), 200), industry), 0)
neutral_multiply_one_wrapper => {'status': 'REJECT', 'reason': 'trivial_neutral_wrapper_of_prior_expression', 'details': None}
  expr: multiply(group_rank(ts_product(group_neutralize(pv13_hierarchy_min2_focused_pureplay_513_sector, subindustry), 200), industr

## 10. Pipeline Torture Harness: Semantic Scorecard Gate

This pounds the expression-level semantic gate with mocked LLM scorecards. It verifies low-score pass, high-score quarantine, malformed-scorecard fail-closed behavior, sign/cross-delay exceptions, medium-score soft quota, portfolio-risk quota pressure, neighbor retrieval, and candidate metadata.


In [19]:
class ScriptedSemanticCritic:
    def __init__(self, scorecards):
        self.scorecards = scorecards
        self.payloads = []

    def score_candidates(self, *, payload):
        self.payloads.append(payload)
        if callable(self.scorecards):
            return {'scorecards': [self.scorecards(row, i) for i, row in enumerate(payload['candidates'])]}
        return {'scorecards': self.scorecards}


def scorecard(candidate_key: str, score: float, *, tags=None, confidence=90, portfolio_risk=20):
    return {
        'candidate_key': candidate_key,
        'overall_similarity_score': score,
        'economic_mechanism_similarity': score,
        'field_role_similarity': score,
        'operator_structure_similarity': score,
        'trading_condition_similarity': score,
        'portfolio_redundancy_risk': portfolio_risk,
        'confidence': confidence,
        'reason': f'mocked semantic score {score}',
        'nearest_matching_mechanism': 'mocked neighbor',
        'semantic_tags': tags or [],
    }


def clone_candidate(expr_suffix: str = ''):
    c = copy.deepcopy(semantic_candidate)
    if expr_suffix:
        c.expression = f'{semantic_candidate.expression} {expr_suffix}'
        c.source['expression'] = c.expression
    c.source = dict(c.source)
    return c


def run_semantic_gate(scorecards, candidates=None, requested_count=None):
    candidates = candidates or [clone_candidate()]
    gate = SemanticNoveltyGate(
        history_index=index,
        target_delay=TARGET_DELAY,
        agent=ScriptedSemanticCritic(scorecards),
    )
    return gate.apply(
        theory=semantic_theory,
        candidates=candidates,
        requested_count=requested_count or len(candidates),
        portfolio_context=portfolio_context,
        history_summary=summary,
        mock=False,
    )

neighbors = retrieve_semantic_neighbors(index, semantic_candidate, target_delay=TARGET_DELAY, limit=12)
print('nearest neighbor count:', len(neighbors))
if neighbors:
    pprint(neighbors[0])
assert len(neighbors) <= 12
assert all('expression' in row and 'score_components' in row for row in neighbors)
assert neighbors == sorted(neighbors, key=lambda r: r['score'], reverse=True)

semantic_cases = {}

low = run_semantic_gate([scorecard('candidate_0', 20)])
semantic_cases['low_similarity_accept'] = low.report
assert len(low.accepted) == 1 and not low.quarantined
assert low.accepted[0].source['semantic_gate_decision'] == 'ACCEPT_LOW_SIMILARITY'
assert low.accepted[0].source['semantic_similarity_score'] == 20

high = run_semantic_gate([scorecard('candidate_0', 92)])
semantic_cases['high_similarity_quarantine'] = high.report
assert len(high.accepted) == 0 and len(high.quarantined) == 1
assert high.quarantined[0]['routing']['decision'] == 'QUARANTINE_HIGH_SIMILARITY'

malformed = run_semantic_gate([{'candidate_key': 'candidate_0'}])
semantic_cases['malformed_scorecard_fail_closed'] = malformed.report
assert len(malformed.accepted) == 0 and len(malformed.quarantined) == 1
assert malformed.quarantined[0]['routing']['decision'] == 'QUARANTINE_HIGH_SIMILARITY'
assert 'missing required' in malformed.quarantined[0]['scorecard']['reason']

sign = run_semantic_gate([scorecard('candidate_0', 92, tags=['sign_repair'])])
semantic_cases['sign_repair_high_similarity_accept'] = sign.report
assert len(sign.accepted) == 1
assert sign.accepted[0].source['semantic_gate_decision'] == 'ACCEPT_SIGN_REPAIR'
assert sign.accepted[0].novelty_score <= 0.25

cross = run_semantic_gate([scorecard('candidate_0', 92, tags=['cross_delay_retest'])])
semantic_cases['cross_delay_high_similarity_accept'] = cross.report
assert len(cross.accepted) == 1
assert cross.accepted[0].cross_delay_retest
assert cross.accepted[0].source['semantic_gate_decision'] == 'ACCEPT_CROSS_DELAY_RETEST'

medium_candidates = [clone_candidate(f'+ 0 /*medium{i}*/') for i in range(5)]
medium = run_semantic_gate(lambda row, i: scorecard(row['candidate_key'], 75), candidates=medium_candidates, requested_count=5)
semantic_cases['medium_similarity_soft_quota'] = medium.report
assert len(medium.accepted) == 2, medium.report
assert len(medium.quarantined) == 3, medium.report
assert all(row.source['semantic_gate_decision'] == 'ACCEPT_SOFT_QUOTA' for row in medium.accepted)
assert all(row['routing']['decision'] == 'QUARANTINE_SOFT_QUOTA_EXCEEDED' for row in medium.quarantined)

risk_candidates = [clone_candidate(f'+ 0 /*risk{i}*/') for i in range(3)]
risk = run_semantic_gate(
    lambda row, i: scorecard(row['candidate_key'], 30, tags=['pool_hurt_neighbor'], confidence=95, portfolio_risk=90),
    candidates=risk_candidates,
    requested_count=3,
)
semantic_cases['low_score_pool_hurt_quota_pressure'] = risk.report
assert len(risk.accepted) == 2
assert len(risk.quarantined) == 1
assert all(row.source['semantic_gate_decision'] == 'ACCEPT_SOFT_QUOTA' for row in risk.accepted)

print('semantic cases:')
pprint(semantic_cases)
STRESS_REPORT['semantic_scorecard_gate'] = semantic_cases


nearest neighbor count: 12
{'admission_decision': None,
 'admission_delta_pool_sharpe': None,
 'admission_global_max_corr': None,
 'admission_reason': None,
 'admission_residual_sharpe': None,
 'alpha_id': 'wpzXvke2',
 'category': 'analyst',
 'dataset_id': 'analyst4',
 'delay': 1,
 'expression': 'rank(ts_delta(if_else(anl4_rd_exp_flag, anl4_rd_exp_flag, '
               'anl4_rd_exp_flag), 5))',
 'failed_checks': ['CONCENTRATED_WEIGHT', 'LOW_FITNESS', 'LOW_SHARPE'],
 'fields': ['anl4_rd_exp_flag'],
 'history_id': 1521,
 'is_fitness': -0.49,
 'is_sharpe': -0.65,
 'is_turnover': 0.5018,
 'max_self_correlation': 0.006942987121362388,
 'operator_skeleton': ['rank', 'ts_delta', 'if_else'],
 'same_delay': True,
 'score': 35.6667,
 'score_components': {'category_match': 5.0,
                      'dataset_match': 10.0,
                      'field_jaccard': 0.0,
                      'same_delay': 4.0,
                      'skeleton_jaccard': 16.666666666666664},
 'search_lane': 'grammar',
 

## 11. Pipeline Torture Harness: Generator Integration And Persistence

This checks that the generator uses the semantic gate as part of the real generation path, writes semantic artifacts, carries semantic metadata into candidate params, and can survive an all-quarantine semantic outcome without producing a bad combined plan.


In [20]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    store_for_generation = TheoryStore(root / 'theories')
    theory_for_generation = mock_theories(n=1, sprint_focus='generator semantic integration')[0]
    store_for_generation.save_theory(theory_for_generation, event_type='created', payload={'source': 'stress_notebook'})
    generator_for_stress = LLMExpressionGenerator(field_csv=field_path, history_index=index, target_delay=TARGET_DELAY)
    records, report = generator_for_stress.generate(
        theory=theory_for_generation,
        store=store_for_generation,
        stage='probe',
        generation_mode='probe',
        max_candidates=6,
        portfolio_context=portfolio_context,
        history_summary=summary,
        field_catalog_summary=field_summary,
        mock=True,
    )
    print('generator records:', len(records))
    pprint(report.get('semantic_gate'))
    assert records, 'mock generation should produce semantic-accepted records'
    assert 'semantic_gate' in report
    for key in ['semantic_request', 'semantic_response', 'semantic_quarantined']:
        path = Path(report['artifact_paths'][key])
        print(key, path, path.exists())
        assert path.exists()
    row = records[0].to_dict()
    semantic_param_keys = [k for k in row['params'] if k.startswith('semantic_')]
    print('semantic param keys:', semantic_param_keys)
    assert 'semantic_similarity_score' in row['params']
    assert 'semantic_gate_decision' in row['params']

class QuarantineAllGate:
    def apply(self, *, theory, candidates, requested_count, portfolio_context, history_summary, mock=False):
        quarantined = [
            {
                'expression': row.expression,
                'scorecard': scorecard(f'candidate_{i}', 99),
                'routing': {'decision': 'QUARANTINE_HIGH_SIMILARITY', 'reason': 'forced quarantine', 'allowed_by_quota': False, 'bucket': 'quarantine'},
                'nearest_neighbors': [],
            }
            for i, row in enumerate(candidates)
        ]
        return SemanticGateResult(
            accepted=[],
            quarantined=quarantined,
            requests=[{'forced': True, 'n_candidates': len(candidates)}],
            responses=[{'candidate_key': f'candidate_{i}', 'routing': q['routing'], 'scorecard': q['scorecard']} for i, q in enumerate(quarantined)],
            report={'requested_count': requested_count, 'accepted_count': 0, 'quarantined_count': len(quarantined), 'decisions': {'QUARANTINE_HIGH_SIMILARITY': len(quarantined)}},
        )

with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    quarantine_store = TheoryStore(root / 'theories')
    quarantine_theory = mock_theories(n=1, sprint_focus='forced quarantine')[0]
    quarantine_store.save_theory(quarantine_theory, event_type='created', payload={'source': 'stress_notebook'})
    quarantine_generator = LLMExpressionGenerator(
        field_csv=field_path,
        history_index=index,
        target_delay=TARGET_DELAY,
        semantic_gate=QuarantineAllGate(),
    )
    quarantine_records, quarantine_report = quarantine_generator.generate(
        theory=quarantine_theory,
        store=quarantine_store,
        stage='probe',
        generation_mode='probe',
        max_candidates=4,
        portfolio_context=portfolio_context,
        history_summary=summary,
        field_catalog_summary=field_summary,
        mock=True,
    )
    print('forced quarantine report:', quarantine_report.get('semantic_gate'))
    assert quarantine_records == []
    assert quarantine_report['semantic_gate']['quarantined_count'] >= 1
    assert Path(quarantine_report['artifact_paths']['semantic_quarantined']).exists()

STRESS_REPORT['generator_integration'] = {
    'normal_records': len(records),
    'normal_semantic_report': report.get('semantic_gate'),
    'forced_quarantine_report': quarantine_report.get('semantic_gate'),
}
print('STRESS REPORT SUMMARY')
pprint(STRESS_REPORT)


generator records: 6
{'accepted_count': 6,
 'decisions': {'ACCEPT_LOW_SIMILARITY': 6},
 'quarantined_count': 0,
 'requested_count': 6,
 'soft_quota_budget': 2}
semantic_request /tmp/tmpjpvtc9f1/theories/analyst_news_confirmation_generator_semantic_i/generation_probe_delay1_20260417_031154_semantic_request.jsonl True
semantic_response /tmp/tmpjpvtc9f1/theories/analyst_news_confirmation_generator_semantic_i/generation_probe_delay1_20260417_031154_semantic_response.jsonl True
semantic_quarantined /tmp/tmpjpvtc9f1/theories/analyst_news_confirmation_generator_semantic_i/generation_probe_delay1_20260417_031154_semantic_quarantined.jsonl True
semantic param keys: ['semantic_similarity_score', 'semantic_similarity_dimensions', 'semantic_similarity_confidence', 'semantic_tags', 'semantic_nearest_neighbors', 'semantic_gate_decision', 'semantic_gate_reason', 'semantic_gate_allowed_by_quota']
forced quarantine report: {'requested_count': 4, 'accepted_count': 0, 'quarantined_count': 4, 'decisions':

## 12. Full Safe Loop: Proposal → Critique → Expression Generation

This runs the complete local loop in a temp theory store. It uses mock proposal/expression generation by default, no Brain submission, and no live LLM call. Use this to inspect object shapes and generated expressions before running the real sprint.


In [21]:
with tempfile.TemporaryDirectory() as td:
    loop_root = Path(td)
    loop_store = TheoryStore(loop_root / 'theories')
    proposed = TheoryProposerAgent().propose(
        history_summary=summary,
        existing_theories=[],
        portfolio_context=portfolio_context,
        target_delay=TARGET_DELAY,
        field_catalog_summary=field_summary,
        sprint_focus='verification dry loop',
        n=3,
        mock=True,
    )
    print('PROPOSED')
    for t in proposed:
        print('-', t.theory_id, '|', t.title)
        loop_store.save_theory(t, event_type='created', payload={'source': 'notebook_dry_loop'})

    dry_critic = DeterministicNoveltyCritic(index, portfolio_context=portfolio_context, target_delay=TARGET_DELAY)
    accepted = []
    print('\nCRITIQUE')
    for t in proposed:
        decision = dry_critic.critique(t)
        print('-', t.theory_id, decision['decision'], '|', decision['reason'][:160])
        loop_store.write_decision(t.theory_id, decision, target_delay=TARGET_DELAY)
        if decision['decision'] == 'ACCEPT':
            accepted.append(t.with_status('ACCEPTED'))

    if not accepted:
        # Keep this notebook diagnostic useful even if the current history is so saturated
        # that the deterministic critic rejects every mock theory. The critique output
        # above remains the real signal; this override only lets us inspect generation shape.
        accepted = [proposed[0].with_status('ACCEPTED_FOR_NOTEBOOK_GENERATION_AUDIT')]
        print('No deterministic accepts; forcing first mock theory into generation audit only.')

    print('\nEXPRESSION GENERATION')
    dry_generator = LLMExpressionGenerator(field_csv=field_path, history_index=index, target_delay=TARGET_DELAY)
    generated_rows = []
    for t in accepted[:2]:
        records, report = dry_generator.generate(
            theory=t,
            store=loop_store,
            stage='probe',
            generation_mode='probe',
            max_candidates=4,
            portfolio_context=portfolio_context,
            history_summary=summary,
            field_catalog_summary=field_summary,
            mock=True,
        )
        print('-', t.theory_id, 'records=', len(records), 'report=', {k: report.get(k) for k in ['accepted_count', 'rejected_count', 'distinct_operator_skeletons']})
        for rec in records[:3]:
            row = rec.to_dict()
            generated_rows.append(row)
            print('  expr:', row['expression'][:220])
            print('  params:', {k: row['params'].get(k) for k in ['mechanism_translation', 'pool_contribution_argument', 'generation_source']})

    assert proposed
    assert accepted
    assert generated_rows
    print('\ndry loop store:', loop_store.theory_dir)


PROPOSED
- analyst_news_confirmation_verification_dry_loo | Analyst revision confirmation by news attention
- fundamental_pv_confirmation_verification_dry_loo | Fundamental quality only when price-volume does not reject it
- options_uncertainty_fundamental_verification_dry_loo | Options-implied uncertainty conditions fundamental surprises

CRITIQUE
- analyst_news_confirmation_verification_dry_loo ACCEPT | passes deterministic novelty gates
- fundamental_pv_confirmation_verification_dry_loo REVISE | overlaps submitted portfolio themes: submitted field exposure: subindustry, submitted field exposure: industry, submitted field exposure: market, submitted fiel
- options_uncertainty_fundamental_verification_dry_loo ACCEPT | passes deterministic novelty gates

EXPRESSION GENERATION
- analyst_news_confirmation_verification_dry_loo records= 4 report= {'accepted_count': 4, 'rejected_count': 0, 'distinct_operator_skeletons': 4}
  expr: group_neutralize(multiply(rank(ts_zscore(eps_estimate_value,

## 13. Optional Live LLM Smoke Tests

Set `RUN_LIVE_LLM = True` in the setup cell to run these. They use your Anthropic key and may cost tokens. They do not submit to Brain.


In [22]:
if RUN_LIVE_LLM:
    os.environ.setdefault('LLM_ECONOMIC_SEARCH_DEBUG_DIR', str(ROOT / 'alpha_discovery.legacy.economic_search_method/artifacts/llm_debug'))
    live_theories = TheoryProposerAgent().propose(
        history_summary=summary,
        existing_theories=store.list_theories(),
        portfolio_context=portfolio_context,
        target_delay=TARGET_DELAY,
        field_catalog_summary=field_summary,
        sprint_focus='verification smoke test: propose two distinct non-duplicative mechanisms',
        n=2,
        mock=False,
    )
    print('live proposer returned:', [t.theory_id for t in live_theories])
    live_reports = {t.theory_id: det_critic.critique(t) for t in live_theories}
    live_decisions = NoveltyCriticAgent().critique(
        theories=live_theories,
        deterministic_reports=live_reports,
        history_summary=summary,
        portfolio_context=portfolio_context,
        target_delay=TARGET_DELAY,
    )
    pprint(live_decisions)
else:
    print('Live LLM smoke test disabled. Set RUN_LIVE_LLM=True to run.')


Live LLM smoke test disabled. Set RUN_LIVE_LLM=True to run.
